In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
Python executable: /opt/anaconda3/bin/python


In [2]:
%pip install pypdf sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pypdf
import sentence_transformers
import sklearn

print("pypdf:", pypdf.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("scikit-learn:", sklearn.__version__)

print("\nAll libraries imported successfully!")

pypdf: 6.19.0
sentence-transformers: 6.1.0
scikit-learn: 1.6.1

All libraries imported successfully!


In [4]:
from pypdf import PdfReader

pdf_path = "data/DS ModelQP with Solutions.pdf"

reader = PdfReader(pdf_path)

print("Total number of pages:", len(reader.pages))

Total number of pages: 44


In [5]:
# Extract text from every page
all_text = ""

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    all_text += page_text + "\n"

print("Text extraction completed!")
print("Total characters extracted:", len(all_text))

Text extraction completed!
Total characters extracted: 52987


In [6]:
print(all_text[:2000])

1 a. Define Distributed Systems. List and explain the significant consequences of Distributed 
Systems 
A Distributed System is a collection of independent computers that appear to users as a single 
coherent system. These computers communicate and coordinate their actions by passing messages 
to achieve a common goal. 
Significant Consequences of Distributed Systems 
Concurrency 
Distributed systems allow multiple components to run concurrently, enabling parallel execution of 
tasks. 
Consequence: Increased performance and scalability but also challenges in synchronization and 
resource sharing. 
Lack of a Global Clock 
Distributed systems do not have a single, universal clock to synchronize all nodes. 
Consequence: Events must be ordered logically using techniques like Lamport timestamps or vector 
clocks, making time coordination complex. 
Independent Failures 
Each component in a distributed system can fail independently. 
Consequence: Fault tolerance mechanisms such as replication

In [7]:
import re

# Replace multiple spaces or tabs with a single space
clean_text = re.sub(r"[ \t]+", " ", all_text)

# Remove excessive blank lines
clean_text = re.sub(r"\n\s*\n+", "\n\n", clean_text)

# Remove leading and trailing whitespace
clean_text = clean_text.strip()

print("Text cleaning completed!")
print("Original characters:", len(all_text))
print("Cleaned characters:", len(clean_text))

Text cleaning completed!
Original characters: 52987
Cleaned characters: 52390


In [8]:
print(clean_text[:2000])

1 a. Define Distributed Systems. List and explain the significant consequences of Distributed 
Systems 
A Distributed System is a collection of independent computers that appear to users as a single 
coherent system. These computers communicate and coordinate their actions by passing messages 
to achieve a common goal. 
Significant Consequences of Distributed Systems 
Concurrency 
Distributed systems allow multiple components to run concurrently, enabling parallel execution of 
tasks. 
Consequence: Increased performance and scalability but also challenges in synchronization and 
resource sharing. 
Lack of a Global Clock 
Distributed systems do not have a single, universal clock to synchronize all nodes. 
Consequence: Events must be ordered logically using techniques like Lamport timestamps or vector 
clocks, making time coordination complex. 
Independent Failures 
Each component in a distributed system can fail independently. 
Consequence: Fault tolerance mechanisms such as replication

In [9]:
# Split the cleaned text into chunks
chunk_size = 1000
overlap = 150

chunks = []

start = 0

while start < len(clean_text):
    end = start + chunk_size
    chunk = clean_text[start:end]

    chunks.append(chunk)

    start += chunk_size - overlap

print("Chunking completed!")
print("Total chunks:", len(chunks))

Chunking completed!
Total chunks: 62


In [10]:
print(chunks[0])

1 a. Define Distributed Systems. List and explain the significant consequences of Distributed 
Systems 
A Distributed System is a collection of independent computers that appear to users as a single 
coherent system. These computers communicate and coordinate their actions by passing messages 
to achieve a common goal. 
Significant Consequences of Distributed Systems 
Concurrency 
Distributed systems allow multiple components to run concurrently, enabling parallel execution of 
tasks. 
Consequence: Increased performance and scalability but also challenges in synchronization and 
resource sharing. 
Lack of a Global Clock 
Distributed systems do not have a single, universal clock to synchronize all nodes. 
Consequence: Events must be ordered logically using techniques like Lamport timestamps or vector 
clocks, making time coordination complex. 
Independent Failures 
Each component in a distributed system can fail independently. 
Consequence: Fault tolerance mechanisms such as replication

In [11]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [12]:
# Convert text chunks into numerical embeddings
embeddings = embedding_model.encode(chunks)

print("Embeddings generated successfully!")
print("Number of chunks:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)

Embeddings generated successfully!
Number of chunks: 62
Embedding dimensions: (62, 384)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(query, top_k=3):
    # Convert the user's question into an embedding
    query_embedding = embedding_model.encode([query])

    # Compare the question with all PDF chunk embeddings
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Get indices of the most similar chunks
    top_indices = similarities.argsort()[::-1][:top_k]

    # Return the matching chunks and their scores
    results = []

    for index in top_indices:
        results.append({
            "chunk": chunks[index],
            "similarity_score": similarities[index]
        })

    return results

print("Similarity search function created!")

Similarity search function created!


In [14]:
query = "Define Distributed Systems and explain its consequences."

results = retrieve_relevant_chunks(query, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Similarity:", round(float(result["similarity_score"]), 4))
    print(result["chunk"])


--- Result 1 ---
Similarity: 0.6541
1 a. Define Distributed Systems. List and explain the significant consequences of Distributed 
Systems 
A Distributed System is a collection of independent computers that appear to users as a single 
coherent system. These computers communicate and coordinate their actions by passing messages 
to achieve a common goal. 
Significant Consequences of Distributed Systems 
Concurrency 
Distributed systems allow multiple components to run concurrently, enabling parallel execution of 
tasks. 
Consequence: Increased performance and scalability but also challenges in synchronization and 
resource sharing. 
Lack of a Global Clock 
Distributed systems do not have a single, universal clock to synchronize all nodes. 
Consequence: Events must be ordered logically using techniques like Lamport timestamps or vector 
clocks, making time coordination complex. 
Independent Failures 
Each component in a distributed system can fail independently. 
Consequence: Fault tol

In [15]:
# Combine the retrieved chunks into one context
context = "\n\n".join(
    result["chunk"] for result in results
)

print("Retrieved context created!")
print("Context characters:", len(context))

Retrieved context created!
Context characters: 3004


In [16]:
print(context)

1 a. Define Distributed Systems. List and explain the significant consequences of Distributed 
Systems 
A Distributed System is a collection of independent computers that appear to users as a single 
coherent system. These computers communicate and coordinate their actions by passing messages 
to achieve a common goal. 
Significant Consequences of Distributed Systems 
Concurrency 
Distributed systems allow multiple components to run concurrently, enabling parallel execution of 
tasks. 
Consequence: Increased performance and scalability but also challenges in synchronization and 
resource sharing. 
Lack of a Global Clock 
Distributed systems do not have a single, universal clock to synchronize all nodes. 
Consequence: Events must be ordered logically using techniques like Lamport timestamps or vector 
clocks, making time coordination complex. 
Independent Failures 
Each component in a distributed system can fail independently. 
Consequence: Fault tolerance mechanisms such as replication

In [19]:
!pip install --upgrade openai httpx2 brotli

In [20]:
import os

print("API key loaded:", bool(os.getenv("sk-")))

API key loaded: False


In [23]:
import os


print("API key loaded:", bool(os.getenv("sk-")))

API key loaded: True


In [24]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client initialized successfully!")

OpenAI client initialized successfully!


In [25]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input="Explain Distributed Systems in one simple sentence."
)

print(response.output_text)

A distributed system is a group of connected computers that work together as if they were one system.


In [26]:
def generate_answer(question, context):
    prompt = f"""
You are a helpful academic assistant.

Answer the question using only the information provided in the context.

If the answer cannot be found in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [27]:
answer = generate_answer(query, context)

print("Final Answer:")
print(answer)

Final Answer:
A **Distributed System** is a collection of independent computers that communicate and coordinate by passing messages, but appear to users as one coherent system working toward a common goal.

### Significant consequences of Distributed Systems

1. **Concurrency**  
   Multiple components can execute tasks at the same time. This improves performance and scalability but creates challenges in synchronization and resource sharing.

2. **Lack of a Global Clock**  
   There is no single universal clock shared by all computers. Therefore, events must be ordered logically using techniques such as Lamport timestamps or vector clocks.

3. **Independent Failures**  
   Components may fail independently of one another. Replication, recovery, and consensus mechanisms are therefore needed to provide fault tolerance, reliability, and availability.

4. **Resource Sharing**  
   Hardware, software, and data can be shared across different locations. Effective resource management is necess

In [28]:
def ask_question(question, top_k=3):
    # Retrieve relevant chunks
    results = retrieve_relevant_chunks(question, top_k=top_k)

    # Combine retrieved chunks into context
    context = "\n\n".join(
        result["chunk"] for result in results
    )

    # Create prompt
    prompt = f"""
You are a helpful academic assistant for Distributed Systems.

Answer the question using only the information provided in the context.

Instructions:
- Give a clear and accurate answer.
- Use simple academic language.
- Preserve important definitions and concepts from the context.
- Use headings or numbered points when appropriate.
- Do not add information that is not supported by the context.
- If the answer is not available in the context, say:
  "I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using OpenAI
    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    # Return answer
    return response.output_text

In [29]:
question = "What is concurrency in distributed systems?"

answer = ask_question(question)

print("Final Answer:")
print(answer)

Final Answer:
## Concurrency in Distributed Systems

Concurrency is the ability of multiple components or processes in a distributed system to run at the same time. It enables the parallel execution of tasks across different computers.

### Consequences

- **Increased performance:** Several tasks can be processed simultaneously.
- **Improved scalability:** The system can handle more work by using multiple components.
- **Synchronization challenges:** Components may need coordination to access shared resources correctly.
- **Resource-sharing challenges:** Concurrent processes may compete for the same data or resources.
